# Batch Non-Interactive Distributed Simulation with pystclient

This example demonstrates how to run a **batch** of non-interactive distributed
simulations using an existing Spring-Mass-Damper (MSD) project.

Two parameter sets are executed in parallel, and the results are collected
and plotted for comparison.


## 1 - Authenticate and select project


In [ ]:
from pystclient.clients import PyStclient

client = PyStclient()
_ = client.authenticate()

# Select an existing project by name
projects = client.project.info_all()
project_info = next(p for p in projects if p.name == "Tutorial")
print(f"Using project: {project_info.name}  (id={project_info.id})")

## 2 - Batch Distributed Simulation
Run **two parameter sets** in parallel as a batch. Each set gets its own
simulator, and the platform schedules them concurrently.


In [ ]:
from pystclient.models import (
    LoggingConfiguration,
    ModelParameters,
    ModelVariable,
    SimulationConfig,
    SimulationParameters,
)
from pystclient.types import SimulationType

# Parameter set 1 - default tolerances
sim_params1 = SimulationParameters(
    base_step_size=0.01,
    end_time=20,
    config_name="Config 1",
    model_parameters=[
        ModelParameters(
            name="Mass1",
            parameters=[ModelVariable(name="mass", initial_value=9)],
        ),
        ModelParameters(
            name="Spring1",
            step_size=0.02,
            parameters=[ModelVariable(name="absTolerance", initial_value=5e-4)],
        ),
        ModelParameters(
            name="Damper1",
            step_size=0.02,
            parameters=[ModelVariable(name="absTolerance", initial_value=5e-4)],
        ),
    ],
)

# Parameter set 2 - larger tolerances and different step size for comparison
sim_params2 = SimulationParameters(
    base_step_size=0.01,
    end_time=20,
    config_name="Config 2",
    model_parameters=[
        ModelParameters(
            name="Mass1",
            step_size=0.02,
            parameters=[ModelVariable(name="absTolerance", initial_value=5e-4)],
        ),
        ModelParameters(
            name="Spring1",
            step_size=0.02,
            parameters=[ModelVariable(name="absTolerance", initial_value=5e-4)],
        ),
        ModelParameters(
            name="Damper1",
            step_size=0.02,
            parameters=[ModelVariable(name="absTolerance", initial_value=5e-4)],
        ),
    ],
)

assert client.project.update_parameters(project_info.id, sim_params1, sim_params2)
print("Parameter sets uploaded: Config 1, Config 2")

# Enable post-plotting so results are persisted
log_config = LoggingConfiguration(post_plotting=True)
assert client.project.update_log_config(project_info.id, log_config)

In [ ]:
import time

# Create a batch of 2 distributed simulators (non-interactive: 2 parameter sets)
batch_statuses, batch_future = client.simulator.create(
    SimulationConfig(
        project_id=project_info.id,
        parameter_set_names=["Config 1", "Config 2"],
        type=SimulationType.DISTRIBUTED,
        batch_size=2,
    )
)

for s in batch_statuses:
    print(f"  Simulator {s.id}  batch_id={s.batch_id}  status={s.status}")

# Wait until all simulators in the batch are ready
assert batch_future.result(600), "Batch simulators did not become ready in time!"
print("All batch simulators are ready.")

simulator_ids = [s.id for s in batch_statuses]

# Poll until every simulator in the batch has finished
while not client.simulator.finished(*simulator_ids):
    time.sleep(2)

print("Batch simulation finished.")

### 2.1 - Retrieve completed batch simulations


In [ ]:
from pystclient.models import SimulationInfo

# Wait for both simulations to appear in the completed list
completed_simulations: list[SimulationInfo] = []

while len(completed_simulations) != len(simulator_ids):
    completed_simulations = client.project.completed_simulations(
        project_id=project_info.id,
        simulator_ids=simulator_ids,
        limit=10,
    )
    time.sleep(5)

for sim in completed_simulations:
    print(f"  Simulation {sim.id}  name={sim.name}  param_set={sim.parameter_set_name}  batch_id={sim.batch_id}")

### 2.2 - Plot batch results side by side


In [ ]:
# pyright: reportUnknownMemberType=false
import matplotlib.pyplot as plt

from pystclient.models import MeasurementQuery, QueryVariable
from pystclient.types import FmuCausalityType
from pystclient.utils.time import convert_to_timestamp

fig, axes = plt.subplots(
    1,
    len(completed_simulations),
    figsize=(7 * len(completed_simulations), 5),
    sharey=True,
)

# Ensure axes is always iterable (handles single-subplot case)
if len(completed_simulations) == 1:
    axes = [axes]

for idx, sim in enumerate(completed_simulations):
    ax = axes[idx]

    # Get measurements for this specific simulation
    sim_measurements = client.measurement.measurements(project_info.id, sim.id)

    if not sim_measurements:
        ax.set_title(f"{sim.parameter_set_name}\n(no data)")
        continue

    # Query the Spring displacement variable
    measurement = sim_measurements[0]
    q = MeasurementQuery(
        variables=[
            QueryVariable(
                instance_name="Spring1",
                name="dis_yx",
                causality=FmuCausalityType.INPUT,
            )
        ],
        time_from=0,
        time_to=10,
    )
    results = client.measurement.query(measurement.id, q)

    for result in results:
        x = convert_to_timestamp(result.x)
        ax.plot(x, result.y)

    ax.set_xlabel("Time [s]")
    ax.set_title(f"{sim.parameter_set_name}")

axes[0].set_ylabel("Displacement [m]")
fig.suptitle("Batch Simulation - Spring Displacement (dis_yx)", fontsize=14)
plt.tight_layout()
plt.show()

### 2.3 - Overlay both batch runs on a single plot


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for sim in completed_simulations:
    sim_measurements = client.measurement.measurements(project_info.id, sim.id)
    if not sim_measurements:
        continue

    measurement = sim_measurements[0]
    q = MeasurementQuery(
        variables=[
            QueryVariable(
                instance_name="Spring1",
                name="dis_yx",
                causality=FmuCausalityType.INPUT,
            )
        ],
        time_from=0,
        time_to=10,
    )
    results = client.measurement.query(measurement.id, q)

    for result in results:
        x = convert_to_timestamp(result.x)
        ax.plot(x, result.y, label=sim.parameter_set_name)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Displacement [m]")
ax.set_title("Batch Comparison - Spring Displacement (dis_yx)")
ax.legend()
plt.tight_layout()
plt.show()